In [2]:
!wget https://files.grouplens.org/datasets/movielens/ml-latest-small.zip

--2026-03-15 00:14:07--  https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 978202 (955K) [application/zip]
Saving to: ‘ml-latest-small.zip’

ml-latest-small.zip 100%[===================>] 955.28K   538KB/s    in 1.8s    

2026-03-15 00:14:13 (538 KB/s) - ‘ml-latest-small.zip’ saved [978202/978202]



In [ ]:
!unzip            ml-latest-small.zip!

In [ ]:
#create hdfs directory  /movie-len using put

In [ ]:
#copy to hdfs at /movie-len

In [2]:
!hdfs dfs -ls /movie-len

2026-03-15 00:19:32,417 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Found 5 items
-rw-r--r--   1 hadoop supergroup       8342 2026-03-15 00:17 /movie-len/README.txt
-rw-r--r--   1 hadoop supergroup     197979 2026-03-15 00:17 /movie-len/links.csv
-rw-r--r--   1 hadoop supergroup     494431 2026-03-15 00:17 /movie-len/movies.csv
-rw-r--r--   1 hadoop supergroup    2483723 2026-03-15 00:17 /movie-len/ratings.csv
-rw-r--r--   1 hadoop supergroup     118660 2026-03-15 00:17 /movie-len/tags.csv


In [ ]:
lines = spark.read.text("data/mllib/als/sample_movielens_ratings.txt").rdd

In [4]:
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS
from pyspark.sql import Row

In [6]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('movie-len').getOrCreate()


In [8]:
spark = SparkSession.builder.appName('movie-len').getOrCreate()


In [27]:
lines = spark.read.csv("hdfs://localhost:9000/movie-len/ratings.csv",header = True, inferSchema = True).rdd

MapPartitionsRDD[64] at javaToPython at NativeMethodAccessorImpl.java:0

In [17]:

df = spark.read.csv("hdfs://localhost:9000/movie-len/ratings.csv", header = True, inferSchema = True)

df.printSchema()

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: integer (nullable = true)



In [31]:

ratingsRDD = lines.map(lambda p: Row(userId=int(p[0]), movieId=int(p[1]),
                                     rating=float(p[2]), timestamp=int(p[3])))

In [32]:
ratings = spark.createDataFrame(ratingsRDD)
(training, test) = ratings.randomSplit([0.8, 0.2])

In [33]:
# Build the recommendation model using ALS on the training data
# Note we set cold start strategy to 'drop' to ensure we don't get NaN evaluation metrics
als = ALS(maxIter=5, regParam=0.01, userCol="userId", itemCol="movieId", ratingCol="rating",
          coldStartStrategy="drop")
model = als.fit(training)

26/03/15 00:47:51 WARN BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeSystemBLAS
26/03/15 00:47:51 WARN BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeRefBLAS
26/03/15 00:47:51 WARN LAPACK: Failed to load implementation from: com.github.fommil.netlib.NativeSystemLAPACK
26/03/15 00:47:51 WARN LAPACK: Failed to load implementation from: com.github.fommil.netlib.NativeRefLAPACK


In [34]:
# Evaluate the model by computing the RMSE on the test data
predictions = model.transform(test)
evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating",
                                predictionCol="prediction")

In [35]:
rmse = evaluator.evaluate(predictions)
print("Root-mean-square error = " + str(rmse))

[Stage 66:====================================================> (193 + 4) / 200]

Root-mean-square error = 1.0781276915238982


In [36]:
# Generate top 10 movie recommendations for each user
userRecs = model.recommendForAllUsers(10)
# Generate top 10 user recommendations for each movie
movieRecs = model.recommendForAllItems(10)

In [38]:
# Generate top 10 movie recommendations for a specified set of users
users = ratings.select(als.getUserCol()).distinct().limit(3)
userSubsetRecs = model.recommendForUserSubset(users, 10)


In [39]:
# Generate top 10 user recommendations for a specified set of movies
movies = ratings.select(als.getItemCol()).distinct().limit(3)
movieSubSetRecs = model.recommendForItemSubset(movies, 10)